# Semantic Document Retrieval with Sentence Embeddings

Retrieves the most relevant Simple English Wikipedia articles for a query using `all-MiniLM-L6-v2` embeddings and cosine similarity.

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.simple",
    split="train[:2000]"
)

documents = [doc["text"] for doc in dataset]

print(f"Loaded {len(documents)} documents.")

Loaded 2000 documents.


In [ ]:
print("Loading embedding model...")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

document_embeddings = model.encode(
    documents,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Embeddings shape:", document_embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Embeddings shape: torch.Size([2000, 384])


In [ ]:
def cosine_similarity(query_embedding, document_embeddings):
    # Normalize embeddings
    query_norm = torch.nn.functional.normalize(query_embedding, p=2, dim=1)
    docs_norm = torch.nn.functional.normalize(document_embeddings, p=2, dim=1)

    # Compute cosine similarity
    similarity_scores = torch.mm(query_norm, docs_norm.T)

    return similarity_scores.squeeze(0)

In [ ]:
def retrieve(query, documents, embeddings, model, top_k=5):

    # Step 1: Encode query
    query_embedding = model.encode(query, convert_to_tensor=True)
    query_embedding = query_embedding.unsqueeze(0)

    # Step 2: Compute similarity
    similarity_scores = cosine_similarity(query_embedding, embeddings)

    # Step 3: Get top_k results
    top_k_indices = torch.topk(similarity_scores, k=top_k).indices

    results = []

    for idx in top_k_indices:
        results.append({
            "document": documents[idx],
            "score": similarity_scores[idx].item()
        })

    return results

In [ ]:
queries = [
    "artificial intelligence",
    "world war history",
    "machine learning"
]

for query in queries:
    print("\n" + "="*60)
    print("Query:", query)
    print("="*60)

    results = retrieve(query, documents, document_embeddings, model, top_k=5)

    for i, result in enumerate(results):
        print(f"\nResult {i+1}")
        print("Similarity Score:", round(result["score"], 4))
        print("Document Preview:", result["document"][:200])


Query: artificial intelligence

Result 1
Similarity Score: 0.4998
Document Preview: Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is a field of study which tries to make computers "smart". They work on their own without being en

Result 2
Similarity Score: 0.4093
Document Preview: The Turing test is a test to see if a computer can interact with a person. The human should not be able to realise it is interacting with a computer. 

Alan Turing thought that if a human could not te

Result 3
Similarity Score: 0.3754
Document Preview: Computer science deals with the theoretical foundations of computation and practical techniques for their application.
Computer science is the science of information. Computer scientists study differe

Result 4
Similarity Score: 0.3359
Document Preview: A computer is a machine that uses electronics to input, process, store, and output data. Data is information such as numbers, words, and lists. Input of 